# GLASS Dedicated Pair Plots

Build publication-count/SI plots and category-driver plots only for GLASS pathogen-antibiotic pairs with available SI data.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_si_analysis_paths,
    build_si_driver_analysis,
    build_yearly_si_dataset,
    load_si_and_clean_glass_list,
    load_yearly_si_inputs,
    prepare_antibiotic_analysis_level,
    save_category_driver_plots,
    save_interesting_si_plots,
)

## Parameters

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PARQUET_OUTPUT_DIR = OUTPUT_DIR / "parquets"
FIGURE_OUTPUT_DIR = OUTPUT_DIR / "figures"
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ASSETS_DIR = PROJECT_ROOT / "assets"
YEARLY_SCREENED_PATH = PARQUET_OUTPUT_DIR / "yearly_amr_screened_pairs.parquet"
UNIFIED_DECISIONS_PATH = PARQUET_OUTPUT_DIR / "yearly_amr_unified_decisions.parquet"
PATHOGEN_BG_PATH = PARQUET_OUTPUT_DIR / "pathogen_tiab_counts.parquet"
ANTIBIOTIC_BG_PATH = PARQUET_OUTPUT_DIR / "antibiotic_tiab_counts.parquet"
ANTIBIOTIC_COARSENING_PATH = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"

GLASS_XLSX_PATH = ASSETS_DIR / "GLASS list.xlsx"
GLASS_CLEANED_PATH = ASSETS_DIR / "glass_amr_cleaned.csv"
BACTERIA_SYNONYM_PATH = ASSETS_DIR / "bacteria_synonym_lookup.csv"
ANTIBIOTIC_SYNONYM_PATH = ASSETS_DIR / "antibiotic_synonym_lookup.csv"

ANTIBIOTIC_ANALYSIS_LEVEL = "antibiotic"  # antibiotic, Class, Simplified categories, High level categories, Origin
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
GLASS_SI_PLOT_DIR = FIGURE_OUTPUT_DIR / "glass_yearly_si_plots"
GLASS_DRIVER_PLOT_DIR = FIGURE_OUTPUT_DIR / "glass_yearly_si_driver_plots"

FIT_START_YEAR = 2000
FIT_END_YEAR = 2023
NUM_FUTURE_YEARS = 5
PLOT_END_YEAR = FIT_END_YEAR + NUM_FUTURE_YEARS

LOESS_FRAC = 0.6
LOESS_DEGREE = 1
TAIL_DAMPING_YEARS = None
FILL_MISSING_AMR_COUNTS_WITH_ZERO = True

CATEGORY_INCLUDE_DECISIONS = ("yes", "uncertain")
DRIVER_LOOKBACK_YEARS = 5
FIGURE_FORMAT = "pdf"
DISPLAY_PLOTS = True
MAX_DISPLAY_PLOTS = 10
MAX_GLASS_PLOTS = None  # set to an int for a quick subset

## Load Inputs

In [ ]:
yearly_screened_df, pathogen_bg_df, antibiotic_bg_df, unified_decisions_df = load_yearly_si_inputs(
    yearly_screened_path=YEARLY_SCREENED_PATH,
    pathogen_background_path=PATHOGEN_BG_PATH,
    antibiotic_background_path=ANTIBIOTIC_BG_PATH,
    unified_decisions_path=UNIFIED_DECISIONS_PATH,
)

yearly_screened_model_df, antibiotic_bg_model_df, unified_decisions_model_df, analysis_summary = prepare_antibiotic_analysis_level(
    yearly_screened_df=yearly_screened_df,
    antibiotic_background_df=antibiotic_bg_df,
    unified_decisions_df=unified_decisions_df,
    coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    analysis_level=ANTIBIOTIC_ANALYSIS_LEVEL,
)

if YEARLY_SI_PATH.exists():
    si_df = pd.read_parquet(YEARLY_SI_PATH)
    print(f"Loaded SI rows: {len(si_df):,} from {YEARLY_SI_PATH}")
else:
    si_df, pooled_model, design_columns = build_yearly_si_dataset(
        yearly_screened_model_df,
        pathogen_bg_df,
        antibiotic_bg_model_df,
        output_path=YEARLY_SI_PATH,
    )
    print(f"Built SI rows: {len(si_df):,} and saved to {YEARLY_SI_PATH}")

_, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

display(glass_pairs_df.head())

## Keep GLASS Pairs With SI Data

In [ ]:
si_pairs_df = si_df[["pathogen", "antibiotic"]].drop_duplicates()
glass_candidate_pairs_df = glass_pairs_df[["pathogen", "antibiotic"]].drop_duplicates()

glass_available_pairs_df = (
    glass_candidate_pairs_df.merge(si_pairs_df, on=["pathogen", "antibiotic"], how="inner")
    .sort_values(["pathogen", "antibiotic"])
    .reset_index(drop=True)
)

latest_counts_df = (
    si_df.sort_values("year")
    .groupby(["pathogen", "antibiotic"], as_index=False)
    .tail(1)[["pathogen", "antibiotic", "year", "count", "SI"]]
    .rename(columns={"year": "latest_year", "count": "latest_count", "SI": "latest_SI"})
)

glass_plot_pairs_df = (
    glass_available_pairs_df.merge(latest_counts_df, on=["pathogen", "antibiotic"], how="left")
    .sort_values(["latest_count", "pathogen", "antibiotic"], ascending=[False, True, True])
    .reset_index(drop=True)
)
if MAX_GLASS_PLOTS is not None:
    glass_plot_pairs_df = glass_plot_pairs_df.head(int(MAX_GLASS_PLOTS)).copy()

print(f"GLASS candidate pairs: {len(glass_candidate_pairs_df):,}")
print(f"GLASS pairs with SI data: {len(glass_available_pairs_df):,}")
print(f"GLASS pairs selected for plotting: {len(glass_plot_pairs_df):,}")
display(glass_plot_pairs_df.head(30))

## Build Category Driver Inputs

In [ ]:
category_counts_df, glass_with_drivers_df = build_si_driver_analysis(
    interesting_df=glass_plot_pairs_df,
    unified_decisions_df=unified_decisions_model_df,
    fit_end_year=FIT_END_YEAR,
    include_decisions=CATEGORY_INCLUDE_DECISIONS,
    lookback_years=DRIVER_LOOKBACK_YEARS,
    output_path=None,
)

driver_pairs_df = glass_plot_pairs_df.merge(
    category_counts_df[["pathogen", "antibiotic"]].drop_duplicates(),
    on=["pathogen", "antibiotic"],
    how="inner",
)

print(f"Category-count rows: {len(category_counts_df):,}")
print(f"GLASS pairs with category-driver data: {len(driver_pairs_df):,}")
display(glass_with_drivers_df.head(30))

## Save Dedicated Plots

In [ ]:
saved_si_plots = save_interesting_si_plots(
    si_df=si_df,
    interesting_df=glass_plot_pairs_df,
    output_dir=GLASS_SI_PLOT_DIR,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    pathogen_background_df=pathogen_bg_df,
    antibiotic_background_df=antibiotic_bg_model_df,
    max_plots=len(glass_plot_pairs_df),
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    fill_missing_amr_counts_with_zero=FILL_MISSING_AMR_COUNTS_WITH_ZERO,
    figure_format=FIGURE_FORMAT,
    display_plots=DISPLAY_PLOTS,
    max_display_plots=MAX_DISPLAY_PLOTS,
)

saved_driver_plots = save_category_driver_plots(
    category_counts_df=category_counts_df,
    si_df=si_df,
    interesting_df=driver_pairs_df,
    output_dir=GLASS_DRIVER_PLOT_DIR,
    start_year=FIT_START_YEAR,
    end_year=PLOT_END_YEAR,
    fit_end_year=FIT_END_YEAR,
    max_plots=len(driver_pairs_df),
    figure_format=FIGURE_FORMAT,
    display_plots=DISPLAY_PLOTS,
    max_display_plots=MAX_DISPLAY_PLOTS,
)

print(f"Saved GLASS SI plots: {saved_si_plots:,} -> {GLASS_SI_PLOT_DIR}")
print(f"Saved GLASS driver plots: {saved_driver_plots:,} -> {GLASS_DRIVER_PLOT_DIR}")